In [ ]:
# Amazon baseline 2.7.2 - Jupyter Optimized
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import logging
from datetime import datetime
from torch_geometric.nn import GATConv  
from torch_geometric.loader import NeighborLoader
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator

# ==========================================
# 0. Logging 配置 (同时兼容 Jupyter 输出)
# ==========================================
def setup_logger(log_name="baseline_amazon_results"):
    if not os.path.exists('logs'):
        os.makedirs('logs')
    timestamp = datetime.now().strftime("%m%d_%H%M")
    log_file = f"logs/{log_name}_{timestamp}.log"
    # 清除之前的 handlers 防止 Jupyter 重复打印
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)
        
    logging.basicConfig(
        level=logging.INFO,
        format='%(message)s',
        handlers=[logging.FileHandler(log_file, mode='w'), logging.StreamHandler()]
    )
    return logging.getLogger()

logger = setup_logger()

# ==========================================
# 1. 实验配置
# ==========================================
class Config:
    hidden_dim = 128
    total_hidden = 1024 
    epochs = 65          
    lr_decay_epoch = 35  
    lr_init = 0.001      
    batch_size = 1024    
    weight_decay = 1e-3  
    seeds = [1, 7, 314, 1227, 2026]

# ==========================================
# 2. 模型定义 (GAT Aligned)
# ==========================================
class AlignedBaseline(torch.nn.Module):
    def __init__(self, in_size, hidden_size, out_size):
        super().__init__()
        self.res_lin = nn.Linear(in_size, hidden_size)
        self.conv1 = GATConv(hidden_size, hidden_size // 8, heads=8, dropout=0.5)
        self.ln1 = nn.LayerNorm(hidden_size)
        self.conv2 = GATConv(hidden_size, hidden_size // 8, heads=8, dropout=0.5)
        self.ln2 = nn.LayerNorm(hidden_size)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 4),
            nn.BatchNorm1d(hidden_size * 4),
            nn.ReLU(),
            nn.Linear(hidden_size * 4, out_size)
        )

    def forward(self, x, edge_index):
        x_proj = self.res_lin(x)
        h = F.elu(self.ln1(self.conv1(x_proj, edge_index)))
        h = F.elu(self.ln2(self.conv2(h, edge_index)) + x_proj)
        logits = self.classifier(h)
        return F.log_softmax(logits, dim=1)

# ==========================================
# 3. 评估函数
# ==========================================
@torch.no_grad()
def run_evaluation(loader, model, device):
    model.eval()
    y_preds, y_trues = [], []
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index)
        y_preds.append(out[:batch.batch_size].cpu())
        y_trues.append(batch.y[:batch.batch_size].cpu())
    
    y_pred_all = torch.cat(y_preds, dim=0).argmax(dim=-1, keepdim=True)
    y_true_all = torch.cat(y_trues, dim=0).view(-1, 1) 
    return y_pred_all, y_true_all

# ==========================================
# 4. 单次实验运行器
# ==========================================
def run_experiment(seed, data, split_idx, dataset_info, device, evaluator):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

    train_loader = NeighborLoader(
        data, num_neighbors=[15, 10], batch_size=Config.batch_size,
        input_nodes=split_idx['train'], shuffle=True, num_workers=4, persistent_workers=True
    )

    model = AlignedBaseline(dataset_info.num_features, Config.total_hidden, dataset_info.num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=Config.lr_init, weight_decay=Config.weight_decay)
    
    best_val_acc = 0.0
    final_test_acc = 0.0

    for epoch in range(1, Config.epochs + 1):
        if epoch == Config.lr_decay_epoch:
            for param_group in optimizer.param_groups:
                param_group['lr'] *= 0.5
            print(f"\n[LR Scheduler] LR decayed to {optimizer.param_groups[0]['lr']}\n")

        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index)
            # 修正标签维度
            loss = F.nll_loss(out[:batch.batch_size], batch.y[:batch.batch_size].squeeze().long())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        
        # --- 每一轮都打印 Loss ---
        if epoch % 5 != 0:
            print(f"Epoch: {epoch:02d} | Loss: {avg_loss:.4f}")

         # --- 每 5 轮评估并打印 Acc ---
        if epoch % 5 == 0:
            # 推理时显存占用极小，直接拉大 batch_size 提速 4-8 倍
            inference_batch_size = 32768 
            
            val_loader = NeighborLoader(
                data, 
                num_neighbors=[15, 10], # 保持与训练一致的感受野以确保精度
                batch_size=inference_batch_size,
                input_nodes=split_idx['valid'], 
                shuffle=False, 
                num_workers=4,           # [加速] 推理也启用多进程采样
                persistent_workers=True
            )
            y_pred_val, y_true_val = run_evaluation(val_loader, model, device)
            val_acc = evaluator.eval({'y_true': y_true_val, 'y_pred': y_pred_val})['acc']

            status = "[ ]"
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                # 仅在 Val 创新高时才跑最耗时的测试集推理
                test_loader = NeighborLoader(
                    data, 
                    num_neighbors=[15, 10], 
                    batch_size=inference_batch_size,
                    input_nodes=split_idx['test'], 
                    shuffle=False, 
                    num_workers=4,
                    persistent_workers=True
                )
                y_pred_test, y_true_test = run_evaluation(test_loader, model, device)
                final_test_acc = evaluator.eval({'y_true': y_true_test, 'y_pred': y_pred_test})['acc']
                status = "[*] NEW BEST"

            # 使用 print 确保在 Jupyter Cell 下方直接显示
            print(f"Epoch: {epoch:02d} | Loss: {avg_loss:.4f} | Val: {val_acc:.4f} | Test: {final_test_acc:.4f} {status}")

    return final_test_acc

# ==========================================
# 5. 主程序入口
# ==========================================
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device detect: {device}")
    
    # 第一次运行会耗时进行数据加载
    dataset = PygNodePropPredDataset(name='ogbn-products', root='data')
    data = dataset[0]
    split_idx = dataset.get_idx_split()
    evaluator = Evaluator(name='ogbn-products')

    results = []
    print("\n" + "="*50)
    print(f"Starting TRUE GAT Aligned Baseline | Seeds: {Config.seeds}")
    print("="*50 + "\n")

    for i, s in enumerate(Config.seeds):
        print(f">>> Run {i+1}/{len(Config.seeds)} (Seed: {s})")
        test_acc = run_experiment(s, data, split_idx, dataset, device, evaluator)
        results.append(test_acc)
        print(f">>> Result for Seed {s}: {test_acc:.4f}\n")

    # 统计计算
    res_array = np.array(results) * 100
    mean_val, std_val = np.mean(res_array), np.std(res_array)

    print("\n" + "=" * 60)
    print("FINAL ALIGNED BASELINE RESULTS")
    print(f"Mean Accuracy: {mean_val:.2f}%")
    print(f"Std Deviation: {std_val:.2f}%")
    print(f"LaTeX Format: {mean_val:.2f} \pm {std_val:.2f}")
    print("=" * 60)

/home/malina/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device detect: cuda


/home/malina/.venv/lib/python3.8/site-packages/ogb/nodeproppred/dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.pro


Starting TRUE GAT Aligned Baseline | Seeds: [1]

>>> Run 1/1 (Seed: 1)


/home/malina/.venv/lib/python3.8/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "


Epoch: 01 | Loss: 0.4677
Epoch: 02 | Loss: 0.3845
Epoch: 03 | Loss: 0.3727
Epoch: 04 | Loss: 0.3877
Epoch: 05 | Loss: 0.3616 | Val: 0.8470 | Test: 0.7030 [*] NEW BEST
Epoch: 06 | Loss: 0.3756
Epoch: 07 | Loss: 0.3592
Epoch: 08 | Loss: 0.3734
Epoch: 09 | Loss: 0.3807
Epoch: 10 | Loss: 0.3646 | Val: 0.8882 | Test: 0.7313 [*] NEW BEST
Epoch: 11 | Loss: 0.3612
Epoch: 12 | Loss: 0.3713


KeyboardInterrupt: 

In [1]:
# Amazon baseline 2.7.2 - GCN Vanilla Baseline (Jupyter Optimized)
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import logging
from datetime import datetime
from torch_geometric.nn import GCNConv  # [GCN MODIFIED] 引入 GCN 算子
from torch_geometric.loader import NeighborLoader
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator

# ==========================================
# 0. Logging 配置
# ==========================================
def setup_logger(log_name="baseline_amazon_gcn_results"): 
    if not os.path.exists('logs'):
        os.makedirs('logs')
    timestamp = datetime.now().strftime("%m%d_%H%M")
    log_file = f"logs/{log_name}_{timestamp}.log"
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)
        
    logging.basicConfig(
        level=logging.INFO,
        format='%(message)s',
        handlers=[logging.FileHandler(log_file, mode='w'), logging.StreamHandler()]
    )
    return logging.getLogger()

logger = setup_logger()

# ==========================================
# 1. 实验配置
# ==========================================
class Config:
    hidden_dim = 128
    total_hidden = 1024 
    epochs = 65          
    lr_decay_epoch = 35  
    lr_init = 0.001      
    batch_size = 1024    
    weight_decay = 1e-3  
    seeds = [1] # 建议先跑 1 个种子，稳定后可改为 [1, 7, 314, 1227, 2026]

# ==========================================
# 2. 模型定义 (GCN Aligned)
# ==========================================
class AlignedBaseline(torch.nn.Module):
    def __init__(self, in_size, hidden_size, out_size):
        super().__init__()
        self.res_lin = nn.Linear(in_size, hidden_size)
        
        # [GCN MODIFIED] 将 GATConv 替换为 GCNConv
        # GCN 不需要 heads 参数，直接使用 hidden_size 以保持宽度对齐 (1024)
        self.conv1 = GCNConv(hidden_size, hidden_size)
        self.ln1 = nn.LayerNorm(hidden_size)
        self.conv2 = GCNConv(hidden_size, hidden_size)
        self.ln2 = nn.LayerNorm(hidden_size)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 4),
            nn.BatchNorm1d(hidden_size * 4),
            nn.ReLU(),
            nn.Linear(hidden_size * 4, out_size)
        )

    def forward(self, x, edge_index):
        x_proj = self.res_lin(x)
        # [GCN MODIFIED] 移除多头相关的维度处理，直接进行 GCN 卷积
        h = F.relu(self.ln1(self.conv1(x_proj, edge_index)))
        h = F.relu(self.ln2(self.conv2(h, edge_index)) + x_proj)
        logits = self.classifier(h)
        return F.log_softmax(logits, dim=1)

# ==========================================
# 3. 评估函数 (保持不变)
# ==========================================
@torch.no_grad()
def run_evaluation(loader, model, device):
    model.eval()
    y_preds, y_trues = [], []
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index)
        y_preds.append(out[:batch.batch_size].cpu())
        y_trues.append(batch.y[:batch.batch_size].cpu())
    
    y_pred_all = torch.cat(y_preds, dim=0).argmax(dim=-1, keepdim=True)
    y_true_all = torch.cat(y_trues, dim=0).view(-1, 1) 
    return y_pred_all, y_true_all

# ==========================================
# 4. 单次实验运行器 (保持加速逻辑与架构不变)
# ==========================================
def run_experiment(seed, data, split_idx, dataset_info, device, evaluator):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

    train_loader = NeighborLoader(
        data, num_neighbors=[15, 10], batch_size=Config.batch_size,
        input_nodes=split_idx['train'], shuffle=True, num_workers=4, persistent_workers=True
    )

    model = AlignedBaseline(dataset_info.num_features, Config.total_hidden, dataset_info.num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=Config.lr_init, weight_decay=Config.weight_decay)
    
    best_val_acc = 0.0
    final_test_acc = 0.0

    for epoch in range(1, Config.epochs + 1):
        if epoch == Config.lr_decay_epoch:
            for param_group in optimizer.param_groups:
                param_group['lr'] *= 0.5
            print(f"\n[LR Scheduler] LR decayed to {optimizer.param_groups[0]['lr']}\n")

        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index)
            # 修正标签维度以匹配 F.nll_loss
            loss = F.nll_loss(out[:batch.batch_size], batch.y[:batch.batch_size].squeeze().long())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        
        # 打印非评估轮次的 Loss
        if epoch % 5 != 0:
            print(f"Epoch: {epoch:02d} | Loss: {avg_loss:.4f}")

        # 每 5 轮执行一次高效率推理
        if epoch % 5 == 0:
            inference_batch_size = 4096 
            
            val_loader = NeighborLoader(
                data, 
                num_neighbors=[15, 10], 
                batch_size=inference_batch_size,
                input_nodes=split_idx['valid'], 
                shuffle=False, 
                num_workers=4,
                persistent_workers=True
            )
            y_pred_val, y_true_val = run_evaluation(val_loader, model, device)
            val_acc = evaluator.eval({'y_true': y_true_val, 'y_pred': y_pred_val})['acc']

            status = "[ ]"
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                # 仅在验证集创新高时进行测试集评估
                test_loader = NeighborLoader(
                    data, 
                    num_neighbors=[15, 10], 
                    batch_size=inference_batch_size,
                    input_nodes=split_idx['test'], 
                    shuffle=False, 
                    num_workers=4,
                    persistent_workers=True
                )
                y_pred_test, y_true_test = run_evaluation(test_loader, model, device)
                final_test_acc = evaluator.eval({'y_true': y_true_test, 'y_pred': y_pred_test})['acc']
                status = "[*] NEW BEST"

            print(f"Epoch: {epoch:02d} | Loss: {avg_loss:.4f} | Val: {val_acc:.4f} | Test: {final_test_acc:.4f} {status}")

    return final_test_acc

# ==========================================
# 5. 主程序入口
# ==========================================
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device detect: {device}")
    
    # 加载 ogbn-products 数据集
    dataset = PygNodePropPredDataset(name='ogbn-products', root='data')
    data = dataset[0]
    split_idx = dataset.get_idx_split()
    evaluator = Evaluator(name='ogbn-products')

    results = []
    print("\n" + "="*50)
    print(f"Starting TRUE GCN Aligned Baseline | Seeds: {Config.seeds}") # [GCN MODIFIED]
    print("="*50 + "\n")

    for i, s in enumerate(Config.seeds):
        print(f">>> Run {i+1}/{len(Config.seeds)} (Seed: {s})")
        test_acc = run_experiment(s, data, split_idx, dataset, device, evaluator)
        results.append(test_acc)
        print(f">>> Result for Seed {s}: {test_acc:.4f}\n")

    # 统计 5 次实验的均值和标准差
    res_array = np.array(results) * 100
    mean_val, std_val = np.mean(res_array), np.std(res_array)

    print("\n" + "=" * 60)
    print("FINAL ALIGNED GCN BASELINE RESULTS") # [GCN MODIFIED]
    print(f"Mean Accuracy: {mean_val:.2f}%")
    print(f"Std Deviation: {std_val:.2f}%")
    print(f"LaTeX Format: {mean_val:.2f} \pm {std_val:.2f}")
    print("=" * 60)

/home/malina/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device detect: cuda


/home/malina/.venv/lib/python3.8/site-packages/ogb/nodeproppred/dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.pro


Starting TRUE GCN Aligned Baseline | Seeds: [1]

>>> Run 1/1 (Seed: 1)


/home/malina/.venv/lib/python3.8/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "


Epoch: 01 | Loss: 0.4454
Epoch: 02 | Loss: 0.3729
Epoch: 03 | Loss: 0.3476
Epoch: 04 | Loss: 0.3273
Epoch: 05 | Loss: 0.3199 | Val: 0.8846 | Test: 0.7355 [*] NEW BEST
Epoch: 06 | Loss: 0.3212
Epoch: 07 | Loss: 0.3239
Epoch: 08 | Loss: 0.3081
Epoch: 09 | Loss: 0.3073
Epoch: 10 | Loss: 0.3011 | Val: 0.8898 | Test: 0.7362 [*] NEW BEST
Epoch: 11 | Loss: 0.3050


KeyboardInterrupt: 

In [1]:
# 2.7 product NS
# ogbn_products baseline NeuralSparse 

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch_geometric.nn import GATConv
from torch_geometric.loader import NeighborLoader
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator

import torch.serialization
torch.serialization.add_safe_globals([set, dict, list, tuple])

# ==========================================
# 1. 实验配置
# ==========================================
class BaseConfig:
    hidden_dim = 128     
    num_heads = 8        
    total_hidden = 1024  
    epochs = 65          # [MODIFIED] 对齐主实验 Epochs
    patience = 30        
    lr_init = 0.001
    batch_size = 1024    # [MODIFIED] 对齐主实验 BatchSize
    target_sparsity = 0.30  
    weight_decay = 1e-3  # [MODIFIED] 对齐主实验 WeightDecay
    seeds = [27]          # [MODIFIED] 先跑 1 个种子测试

# ==========================================
# 2. NeuralSparse 组件
# ==========================================
class NeuralSparse_Sampler(nn.Module):
    def __init__(self, h_dim):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(h_dim * 2, h_dim // 2),
            nn.ReLU(),
            nn.Linear(h_dim // 2, 1)
        )

    def forward(self, h, edge_index, tau=1.0, hard=True):
        row, col = edge_index
        edge_feat = torch.cat([h[row], h[col]], dim=-1)
        logits = self.scorer(edge_feat).squeeze()
        sampling_logits = torch.stack([torch.zeros_like(logits), logits], dim=-1)
        weights = F.gumbel_softmax(sampling_logits, tau=tau, hard=hard, dim=-1)[:, 1]
        return weights, logits

class NeuralSparse_System(torch.nn.Module):
    def __init__(self, in_size, hidden_size, out_size):
        super().__init__()
        self.total_hidden = hidden_size * 8
        self.res_lin = nn.Linear(in_size, self.total_hidden)
        
        self.gat1 = GATConv(self.total_hidden, hidden_size, heads=8, dropout=0.5)
        self.bn1 = nn.BatchNorm1d(self.total_hidden)
        self.gat2 = GATConv(self.total_hidden, hidden_size, heads=8, dropout=0.5)
        self.bn2 = nn.BatchNorm1d(self.total_hidden)
        
        self.sampler_net = NeuralSparse_Sampler(self.total_hidden)
        
        self.classifier = nn.Sequential(
            nn.Linear(self.total_hidden, hidden_size * 4),
            nn.BatchNorm1d(hidden_size * 4),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden_size * 4, out_size)
        )

    def forward(self, x, edge_index, tau=1.0, hard=True):
        x_proj = self.res_lin(x)
        h1 = F.elu(self.bn1(self.gat1(x_proj, edge_index)))
        h_base = F.elu(self.bn2(self.gat2(h1, edge_index)))
        
        weights, logits_raw = self.sampler_net(h_base, edge_index, tau=tau, hard=hard)
        
        row, col = edge_index
        # 稀疏聚合：只保留被采样的边
        h_sparse = h_base + (weights.view(-1, 1) * h_base[col]).new_zeros(h_base.shape).index_add_(0, row, weights.view(-1, 1) * h_base[col])
        
        logits = self.classifier(h_sparse)
        return F.log_softmax(logits, dim=1), weights, logits_raw

# ==========================================
# 3. 训练与评估逻辑
# ==========================================
def train_step(model, loader, optimizer, device, epoch):
    model.train()
    total_loss = 0
    tau = max(0.2, 1.0 - (epoch / 30)) # [MODIFIED] 调整退火步长
    
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        log_probs, weights, _ = model(batch.x, batch.edge_index, tau=tau, hard=(epoch > 30))
        
        loss_clf = F.nll_loss(log_probs[:batch.batch_size], batch.y.squeeze()[:batch.batch_size].long())
        loss_sp = F.l1_loss(weights.mean(), torch.tensor(BaseConfig.target_sparsity, device=device))
        
        loss = loss_clf + 1.0 * loss_sp
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, device, evaluator):
    model.eval()
    y_preds, y_trues = [], []
    for batch in loader:
        batch = batch.to(device)
        # 推理时必须使用 hard=True
        lp, _, _ = model(batch.x, batch.edge_index, hard=True)
        y_preds.append(lp[:batch.batch_size].argmax(dim=-1).cpu())
        y_trues.append(batch.y[:batch.batch_size].cpu())
        
    y_pred = torch.cat(y_preds, dim=0).numpy().reshape(-1, 1)
    y_true = torch.cat(y_trues, dim=0).numpy().reshape(-1, 1)
    return evaluator.eval({'y_true': y_true, 'y_pred': y_pred})['acc']

# ==========================================
# 4. 实验主循环
# ==========================================
def run_experiment(seed, device):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    
    # [MODIFIED] 修改为 ogbn-products
    dataset = PygNodePropPredDataset(name='ogbn-products', root='./data', transform=T.ToUndirected())
    data = dataset[0]
    split_idx = dataset.get_idx_split()
    evaluator = Evaluator(name='ogbn-products')

    # [MODIFIED] 训练采样对齐 [15, 10]
    train_loader = NeighborLoader(data, num_neighbors=[15, 10], batch_size=BaseConfig.batch_size,
                                  input_nodes=split_idx['train'], shuffle=True, num_workers=4, persistent_workers=True)
    
    model = NeuralSparse_System(dataset.num_features, BaseConfig.hidden_dim, dataset.num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=BaseConfig.lr_init, weight_decay=BaseConfig.weight_decay)

    best_val, final_test, stagnant = 0, 0, 0
    
    for epoch in range(1, BaseConfig.epochs + 1):
        loss = train_step(model, train_loader, optimizer, device, epoch)
        
        # [MODIFIED] Jupyter 打印适配
        if epoch % 5 != 0:
            print(f"Epoch: {epoch:02d} | Loss: {loss:.4f}")

        if epoch % 5 == 0:
            # [MODIFIED] 评估加速配置：大 Batch + 开启多进程
            inference_batch_size = 4096
            val_loader = NeighborLoader(data, num_neighbors=[15, 10], batch_size=inference_batch_size,
                                         input_nodes=split_idx['valid'], shuffle=False, num_workers=4)
            
            val_acc = evaluate(model, val_loader, device, evaluator)
            
            status = "[ ]"
            if val_acc > best_val:
                best_val = val_acc
                test_loader = NeighborLoader(data, num_neighbors=[15, 10], batch_size=inference_batch_size,
                                              input_nodes=split_idx['test'], shuffle=False, num_workers=4)
                final_test = evaluate(model, test_loader, device, evaluator)
                status = "[*] NEW BEST"
                torch.save(model.state_dict(), 'best_neural_sparse.pt')
            
            else:
                stagnant += 5
            
            print(f"Epoch: {epoch:02d} | Loss: {loss:.4f} | Val: {val_acc:.4f} | Test: {final_test:.4f} {status}")
            if stagnant >= BaseConfig.patience: break
            
    return final_test

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    results = []
    print(f"Starting Baseline: NeuralSparse on Products (Target Sparsity: {BaseConfig.target_sparsity})")
    for s in BaseConfig.seeds:
        acc = run_experiment(s, device)
        results.append(acc)
    
    print(f"\nFinal NeuralSparse Results: {np.mean(results)*100:.2f} ± {np.std(results)*100:.2f}")

/home/malina/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting Baseline: NeuralSparse on Products (Target Sparsity: 0.3)


/home/malina/.venv/lib/python3.8/site-packages/ogb/nodeproppred/dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.pro

Epoch: 01 | Loss: 0.6509
Epoch: 02 | Loss: 0.4733
Epoch: 03 | Loss: 0.4654
Epoch: 04 | Loss: 0.4737
Epoch: 05 | Loss: 0.4921 | Val: 0.8587 | Test: 0.7056 [*] NEW BEST
Epoch: 06 | Loss: 0.4801
Epoch: 07 | Loss: 0.4788
Epoch: 08 | Loss: 0.4737
Epoch: 09 | Loss: 0.4793
Epoch: 10 | Loss: 0.4918 | Val: 0.8548 | Test: 0.7056 [ ]
Epoch: 11 | Loss: 0.4876


KeyboardInterrupt: 

In [1]:
# 2.7  版本 baseline - Random DropEdge (ogbn-products 对齐)

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch_geometric.nn import GATConv
from torch_geometric.loader import NeighborLoader
from torch_geometric.utils import dropout_edge
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator

# [MODIFIED] 修复 PyTorch 2.4+ 安全加载机制导致的 OGB 读取报错
import torch.serialization
torch.serialization.add_safe_globals([set, dict, list, tuple])

# ==========================================
# 1. 实验配置
# ==========================================
class BaseConfig:
    hidden_dim = 128     
    num_heads = 8        
    total_hidden = 1024  
    epochs = 65          # [MODIFIED] 对齐主实验总轮数
    patience = 30        
    lr_init = 0.001
    batch_size = 1024    # [MODIFIED] 对齐主实验训练 BatchSize
    weight_decay = 1e-3  # [MODIFIED] 对齐主实验权重衰减
    target_sparsity = 0.30  
    seeds = [1]          # [MODIFIED] 先运行 1 个种子，稳定后可改为 [1, 7, 314, 1227, 2026]

# ==========================================
# 2. 对齐型 GAT 模型
# ==========================================
class GAT_DropEdge(torch.nn.Module):
    def __init__(self, in_size, head_dim, num_heads, out_size):
        super().__init__()
        self.res_lin = nn.Linear(in_size, head_dim * num_heads)
        self.conv1 = GATConv(head_dim * num_heads, head_dim, heads=num_heads, dropout=0.5)
        self.bn1 = nn.BatchNorm1d(head_dim * num_heads)
        self.conv2 = GATConv(head_dim * num_heads, head_dim, heads=num_heads, dropout=0.5)
        self.bn2 = nn.BatchNorm1d(head_dim * num_heads)
        
        self.classifier = nn.Sequential(
            nn.Linear(head_dim * num_heads, (head_dim * num_heads) // 2),
            nn.BatchNorm1d((head_dim * num_heads) // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear((head_dim * num_heads) // 2, out_size)
        )

    def forward(self, x, edge_index):
        x_proj = self.res_lin(x)
        x = F.elu(self.bn1(self.conv1(x_proj, edge_index)))
        x = F.elu(self.bn2(self.conv2(x, edge_index))) + x_proj
        logits = self.classifier(x)
        return F.log_softmax(logits, dim=1)

# ==========================================
# 3. Mini-batch 评估函数
# ==========================================
@torch.no_grad()
def inference_minibatch(model, loader, device, evaluator):
    model.eval()
    y_preds, y_trues = [], []
    for batch in loader:
        batch = batch.to(device)
        # 推理阶段同样维持 30% 稀疏度
        edge_index, _ = dropout_edge(batch.edge_index, p=1 - BaseConfig.target_sparsity)
        
        out = model(batch.x, edge_index)
        y_preds.append(out[:batch.batch_size].cpu())
        y_trues.append(batch.y[:batch.batch_size].cpu())
        
    y_pred = torch.cat(y_preds, dim=0).argmax(dim=-1, keepdim=True).numpy()
    y_true = torch.cat(y_trues, dim=0).numpy()
    return evaluator.eval({'y_true': y_true, 'y_pred': y_pred})['acc']

# ==========================================
# 4. 实验运行器
# ==========================================
def run_experiment(seed, device):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    
    # [MODIFIED] 修改数据集为 ogbn-products
    dataset = PygNodePropPredDataset(name='ogbn-products', root='./data', transform=T.ToUndirected())
    data = dataset[0]
    split_idx = dataset.get_idx_split()
    evaluator = Evaluator(name='ogbn-products')

    # [MODIFIED] 训练采样数对齐主实验 [15, 10]
    train_loader = NeighborLoader(data, num_neighbors=[15, 10], batch_size=BaseConfig.batch_size,
                                  input_nodes=split_idx['train'], shuffle=True, num_workers=4, persistent_workers=True)
    
    model = GAT_DropEdge(dataset.num_features, BaseConfig.hidden_dim, BaseConfig.num_heads, dataset.num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=BaseConfig.lr_init, weight_decay=BaseConfig.weight_decay)

    best_val_acc, final_test_acc, stagnant_count = 0, 0, 0

    for epoch in range(1, BaseConfig.epochs + 1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            # [DropEdge 核心逻辑]
            edge_index, _ = dropout_edge(batch.edge_index, p=1 - BaseConfig.target_sparsity)
            
            out = model(batch.x, edge_index)
            loss = F.nll_loss(out[:batch.batch_size], batch.y.squeeze()[:batch.batch_size].long())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # [MODIFIED] Jupyter 每一轮打印进度
        if epoch % 5 != 0:
            print(f"Epoch: {epoch:02d} | Loss: {total_loss/len(train_loader):.4f}")

        if epoch % 5 == 0:
            # [MODIFIED] 评估加速配置：大 Batch + 采样深度对齐 [15, 10]
            inf_batch_size = 4096
            val_loader = NeighborLoader(data, num_neighbors=[15, 10], batch_size=inf_batch_size,
                                         input_nodes=split_idx['valid'], shuffle=False, num_workers=4)
            
            val_acc = inference_minibatch(model, val_loader, device, evaluator)
            
            status = "[ ]"
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                test_loader = NeighborLoader(data, num_neighbors=[15, 10], batch_size=inf_batch_size,
                                              input_nodes=split_idx['test'], shuffle=False, num_workers=4)
                final_test_acc = inference_minibatch(model, test_loader, device, evaluator)
                status = "[*] NEW BEST"
                torch.save(model.state_dict(), f'best_dropedge_seed{seed}.pt')
            else:
                stagnant_count += 5
                status = "[ ]"
            
            print(f"Seed: {seed:4d} | Ep: {epoch:02d} | Loss: {total_loss/len(train_loader):.4f} | Val: {val_acc:.4f} | Test: {final_test_acc:.4f} {status}")
            if stagnant_count >= BaseConfig.patience: break
            
    return final_test_acc

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    results = []
    print(f"Starting Baseline: GAT + Random DropEdge on Amazon-Products (Sparsity: {BaseConfig.target_sparsity})")
    for s in BaseConfig.seeds:
        acc = run_experiment(s, device)
        results.append(acc)
    
    print("\n" + "="*40)
    print(f"Final GAT-DropEdge Results: {np.mean(results)*100:.2f} ± {np.std(results)*100:.2f}")
    print("="*40)

/home/malina/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting Baseline: GAT + Random DropEdge on Amazon-Products (Sparsity: 0.3)


/home/malina/.venv/lib/python3.8/site-packages/ogb/nodeproppred/dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.pro

Epoch: 01 | Loss: 0.8391
Epoch: 02 | Loss: 0.6539
Epoch: 03 | Loss: 0.6398
Epoch: 04 | Loss: 0.6521
Seed:    1 | Ep: 05 | Loss: 0.6679 | Val: 0.8111 | Test: 0.6413 [*] NEW BEST
Epoch: 06 | Loss: 0.6593
Epoch: 07 | Loss: 0.6648
Epoch: 08 | Loss: 0.6635
Epoch: 09 | Loss: 0.6898
Seed:    1 | Ep: 10 | Loss: 0.6622 | Val: 0.8235 | Test: 0.6507 [*] NEW BEST
Epoch: 11 | Loss: 0.6633
Epoch: 12 | Loss: 0.6740
Epoch: 13 | Loss: 0.6539
Epoch: 14 | Loss: 0.6555
Seed:    1 | Ep: 15 | Loss: 0.6554 | Val: 0.8140 | Test: 0.6507 [ ]
Epoch: 16 | Loss: 0.6640
Epoch: 17 | Loss: 0.6723


KeyboardInterrupt: 

In [ ]:
# 2.7 ogbn-products baseline top-k pruning (TAP)

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch_geometric.nn import GATConv
from torch_geometric.loader import NeighborLoader
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator

# [MODIFIED] 针对 PyTorch 2.4+ 的安全加载修复
import torch.serialization
torch.serialization.add_safe_globals([set, dict, list, tuple])

# ==========================================
# 1. 实验配置 (对齐 PANS 主实验)
# ==========================================
class BaseConfig:
    hidden_dim = 128     
    num_heads = 8        
    total_hidden = 1024  
    epochs = 65          # [MODIFIED] 对齐主实验 65 轮
    patience = 30        
    lr_init = 0.001
    batch_size = 1024    # [MODIFIED] 对齐主实验 Batch Size
    weight_decay = 1e-3  # [MODIFIED] 对齐主实验 WD
    target_sparsity = 0.30  
    seeds = [1]          # [MODIFIED] 先运行 1 个种子，稳定后可改为全量种子列表

# ==========================================
# 2. 剪枝模型
# ==========================================
class GAT_Pruning(torch.nn.Module):
    def __init__(self, in_size, head_dim, num_heads, out_size):
        super().__init__()
        self.res_lin = nn.Linear(in_size, head_dim * num_heads)
        
        self.conv1 = GATConv(head_dim * num_heads, head_dim, heads=num_heads, dropout=0.5)
        self.bn1 = nn.BatchNorm1d(head_dim * num_heads)
        
        self.conv2 = GATConv(head_dim * num_heads, head_dim, heads=num_heads, dropout=0.5)
        self.bn2 = nn.BatchNorm1d(head_dim * num_heads)
        
        self.classifier = nn.Sequential(
            nn.Linear(head_dim * num_heads, (head_dim * num_heads) // 2),
            nn.BatchNorm1d((head_dim * num_heads) // 2),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear((head_dim * num_heads) // 2, out_size)
        )

    def prune_edges(self, edge_index, edge_attr, sparsity):
        if edge_attr is None or edge_index.numel() == 0:
            return edge_index
        
        # [ALIGNED] 取 Attention Heads 的平均分作为剪枝依据
        score = edge_attr.mean(dim=-1)
        num_edges = edge_index.size(1)
        
        k = int(num_edges * sparsity)
        k = max(1, min(k, num_edges))
        
        _, top_indices = torch.topk(score, k, sorted=False)
        return edge_index[:, top_indices]

    def forward(self, x, edge_index):
        x_proj = self.res_lin(x)
        
        # 第一层卷积并获取 Attention 权重
        x, (edge_index_tmp, alpha) = self.conv1(x_proj, edge_index, return_attention_weights=True)
        x = F.elu(self.bn1(x))
        
        # 基于第一层 Attention 进行 Top-K 剪枝
        pruned_edge_index = self.prune_edges(edge_index_tmp, alpha, BaseConfig.target_sparsity)
        
        # 第二层聚合 (在剪枝后的拓扑上运行)
        x = self.conv2(x, pruned_edge_index)
        x = F.elu(self.bn2(x)) + x_proj
        
        logits = self.classifier(x)
        return F.log_softmax(logits, dim=1)

# ==========================================
# 3. 推理评估函数 (Jupyter 加速版)
# ==========================================
@torch.no_grad()
def inference_minibatch(model, loader, device, evaluator):
    model.eval()
    y_preds, y_trues = [], []
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index)
        y_preds.append(out[:batch.batch_size].argmax(dim=-1).cpu())
        y_trues.append(batch.y[:batch.batch_size].cpu())
        
    y_pred = torch.cat(y_preds, dim=0).view(-1, 1).numpy()
    y_true = torch.cat(y_trues, dim=0).view(-1, 1).numpy()
    return evaluator.eval({'y_true': y_true, 'y_pred': y_pred})['acc']

# ==========================================
# 4. 实验运行
# ==========================================
def run_experiment(seed, device):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    
    # [MODIFIED] 切换至 products 数据集
    dataset = PygNodePropPredDataset(name='ogbn-products', root='data', transform=T.ToUndirected())
    data = dataset[0]
    split_idx = dataset.get_idx_split()
    evaluator = Evaluator(name='ogbn-products')

    # [MODIFIED] 数据加载对齐 PANS (4 Workers + 15,10 Neighbors)
    train_loader = NeighborLoader(data, num_neighbors=[15, 10], batch_size=BaseConfig.batch_size,
                                  input_nodes=split_idx['train'], shuffle=True, num_workers=4, persistent_workers=True)
    
    model = GAT_Pruning(dataset.num_features, BaseConfig.hidden_dim, BaseConfig.num_heads, dataset.num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=BaseConfig.lr_init, weight_decay=BaseConfig.weight_decay)

    best_val_acc, final_test_acc, stagnant_count = 0, 0, 0

    print(f"\n>>> Seed: {seed} | Starting Training...")

    for epoch in range(1, BaseConfig.epochs + 1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index)
            # 对齐 OGB 标签计算
            loss = F.nll_loss(out[:batch.batch_size], batch.y.squeeze()[:batch.batch_size].long())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # [MODIFIED] 每轮打印 Loss，每 5 轮评估
        if epoch % 5 != 0:
            print(f"Epoch: {epoch:02d} | Loss: {total_loss/len(train_loader):.4f}")

        if epoch % 5 == 0:
            # [MODIFIED] 评估加速配置：大 Batch (32768) + 采样深度对齐
            inf_batch_size = 4096
            val_loader = NeighborLoader(data, num_neighbors=[15, 10], batch_size=inf_batch_size,
                                         input_nodes=split_idx['valid'], shuffle=False, num_workers=4)
            
            val_acc = inference_minibatch(model, val_loader, device, evaluator)
            
            status = "[ ]"
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                test_loader = NeighborLoader(data, num_neighbors=[15, 10], batch_size=inf_batch_size,
                                              input_nodes=split_idx['test'], shuffle=False, num_workers=4)
                final_test_acc = inference_minibatch(model, test_loader, device, evaluator)
                status = "[*] NEW BEST"
                torch.save(model.state_dict(), f'best_tap_seed{seed}.pt')
            else:
                stagnant_count += 5
            
            print(f"Epoch: {epoch:02d} | Loss: {total_loss/len(train_loader):.4f} | Val: {val_acc:.4f} | Test: {final_test_acc:.4f} {status}")
            if stagnant_count >= BaseConfig.patience: break
            
    return final_test_acc

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    results = []
    print(f"Starting Baseline: TAP (Top-K Pruning) on Amazon-Products (Sparsity: {BaseConfig.target_sparsity})")
    for s in BaseConfig.seeds:
        acc = run_experiment(s, device)
        results.append(acc)
    
    print("\n" + "="*45)
    print(f"Final TAP Results: {np.mean(results)*100:.2f} ± {np.std(results)*100:.2f}")
    print("="*45)

/home/malina/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting Baseline: TAP (Top-K Pruning) on Amazon-Products (Sparsity: 0.3)


/home/malina/.venv/lib/python3.8/site-packages/ogb/nodeproppred/dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.pro


>>> Seed: 1 | Starting Training...
Epoch: 01 | Loss: 0.9010
Epoch: 02 | Loss: 0.6782
Epoch: 03 | Loss: 0.6418
Epoch: 04 | Loss: 0.6397
Epoch: 05 | Loss: 0.6631 | Val: 0.7529 | Test: 0.5881 [*] NEW BEST
Epoch: 06 | Loss: 0.6793
Epoch: 07 | Loss: 0.6426
Epoch: 08 | Loss: 0.6471
Epoch: 09 | Loss: 0.6649


In [1]:
# GSAT baseline

# ogbn-products Baseline: Graph Stochastic Attention (GSAT)
import os
import os.path as osp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.utils import add_self_loops
from torch_geometric.loader import NeighborLoader 
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator 
import numpy as np

# ==========================================
# 1. Global 参数配置
# ==========================================
class Config:
    hidden_dim = 128     
    proj_dim = 256       
    epochs = 100         # [MODIFIED] Products 数据集大，收敛较快
    soft_end = 40        
    lr_init = 0.001      
    batch_size = 1024    # [MODIFIED] 增加 Batch Size 提高吞吐量
    weight_decay = 5e-4    
    patience = 10        # [MODIFIED] 适当增加容忍度
    # GSAT 核心超参
    beta = 0.3            
    r = 0.3               

# ==========================================
# 2. GSAT 核心架构组件 (保持逻辑不变)
# ==========================================

class GSAT_Sampler(nn.Module):
    def __init__(self, in_dim, proj_dim=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim * 2, proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, 1)
        )
        
    def forward(self, h_i, h_j, tau=1.0, training=True):
        edge_feat = torch.cat([h_i, h_j], dim=-1)
        logits = self.mlp(edge_feat).squeeze(-1) 
        sampling_logits = torch.stack([torch.zeros_like(logits), logits], dim=-1)
        
        if training:
            weights = F.gumbel_softmax(sampling_logits, tau=tau, hard=True, dim=-1)[:, 1]
        else:
            weights = (torch.sigmoid(logits) > 0.5).float()
        return weights, logits

class GSATSystem(torch.nn.Module):
    def __init__(self, in_size, hidden_size, out_size):
        super().__init__()
        self.total_hidden = hidden_size * 8
        self.res_lin = nn.Linear(in_size, self.total_hidden)
        
        self.gnn_shared = nn.ModuleList([
            GATConv(self.total_hidden, hidden_size, heads=8, dropout=0.3), # [MODIFIED] 调低 Dropout 增加稳定性
            GATConv(self.total_hidden, hidden_size, heads=8, dropout=0.3)
        ])
        self.bns = nn.ModuleList([nn.BatchNorm1d(self.total_hidden) for _ in range(2)])
        self.sampler_net = GSAT_Sampler(in_dim=self.total_hidden)
        
        self.classifier = nn.Sequential(
            nn.Linear(self.total_hidden, hidden_size * 4),
            nn.BatchNorm1d(hidden_size * 4),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden_size * 4, out_size)
        )

    def forward(self, x, edge_index, tau=1.0):
        h = self.res_lin(x)
        for i, conv in enumerate(self.gnn_shared):
            h = F.elu(self.bns[i](conv(h, edge_index)))
        
        weights, logits_raw = self.sampler_net(h[edge_index[0]], h[edge_index[1]], tau=tau, training=self.training)
        row, col = edge_index
        h_s = torch.zeros_like(h).index_add_(0, row, weights.view(-1, 1) * h[col])
        logits = self.classifier(h_s + h) 
        return F.log_softmax(logits, dim=1), weights, logits_raw

# ==========================================
# 3. 核心训练函数 (逻辑不变)
# ==========================================

def get_kl_loss(p_uv_logits, r):
    p = torch.sigmoid(p_uv_logits)
    kl = p * torch.log(p / r + 1e-9) + (1 - p) * torch.log((1 - p) / (1 - r) + 1e-9)
    return kl.mean()

def train_minibatch(epoch):
    model.train()
    total_loss_epoch, total_clf_loss, total_kl_loss = 0, 0, 0
    tau = max(0.2, 1.0 - (epoch / 50)) 

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        log_probs, weights, logits_raw = model(batch.x, batch.edge_index, tau=tau)
        loss_clf = F.nll_loss(log_probs[:batch.batch_size], batch.y.squeeze()[:batch.batch_size])
        loss_kl = get_kl_loss(logits_raw, Config.r)
        total_loss = loss_clf + Config.beta * loss_kl
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss_epoch += total_loss.item()
        total_clf_loss += loss_clf.item()
        total_kl_loss += loss_kl.item()

    return total_loss_epoch/len(train_loader), total_clf_loss/len(train_loader), total_kl_loss/len(train_loader)

@torch.no_grad()
def inference_minibatch(loader):
    model.eval() 
    y_preds, y_trues = [], []
    for batch in loader:
        batch = batch.to(device)
        log_probs, _, _ = model(batch.x, batch.edge_index)
        y_preds.append(log_probs[:batch.batch_size].argmax(dim=-1).cpu())
        y_trues.append(batch.y[:batch.batch_size].cpu())
        
    y_pred = torch.cat(y_preds, dim=0).numpy().reshape(-1, 1)
    y_true = torch.cat(y_trues, dim=0).numpy().reshape(-1, 1)
    return evaluator.eval({'y_true': y_true, 'y_pred': y_pred})['acc']

# ==========================================
# 4. 数据加载与主循环 (针对 Products 修改)
# ==========================================

# [MODIFIED] 修改数据集名称为 ogbn-products
dataset = PygNodePropPredDataset(name='ogbn-products', root='data')
data = dataset[0]

# [MODIFIED] Products 节点标签通常是 long 型，确保转换正确
data.y = data.y.to(torch.long)

# 保持自环添加逻辑
edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)
data.edge_index = edge_index

split_idx = dataset.get_idx_split()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# [MODIFIED] Products 邻居采样数调整：由于图更稠密，适当增加采样数
train_loader = NeighborLoader(data, num_neighbors=[25, 20], batch_size=Config.batch_size, 
                              input_nodes=split_idx['train'], shuffle=True, num_workers=4)

# [MODIFIED] Products 验证和测试不能用 -1 (全邻居)，会导致 OOM，需固定采样数
valid_loader = NeighborLoader(data, num_neighbors=[20, 15], batch_size=Config.batch_size, 
                              input_nodes=split_idx['valid'], shuffle=False, num_workers=4)
test_loader = NeighborLoader(data, num_neighbors=[20, 15], batch_size=Config.batch_size, 
                             input_nodes=split_idx['test'], shuffle=False, num_workers=4)

# 初始化
model = GSATSystem(dataset.num_features, Config.hidden_dim, dataset.num_classes).to(device)

# 差分学习率
sampler_params = list(model.sampler_net.parameters())
other_params = [p for n, p in model.named_parameters() if 'sampler_net' not in n]

optimizer = torch.optim.Adam([
    {'params': other_params, 'lr': Config.lr_init}, 
    {'params': sampler_params, 'lr': Config.lr_init * 0.1}
], weight_decay=Config.weight_decay)

# [MODIFIED] 修改评估器名称
evaluator = Evaluator(name='ogbn-products')
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 70], gamma=0.5)

history = {'train_loss': [], 'clf_loss': [], 'kl_loss': [], 'val_acc': [], 'test_acc': [], 'sparsity': []}
best_valid_acc, stagnant_epochs = 0.0, 0

print(f"Starting GSAT on Products: Beta: {Config.beta}, R: {Config.r}")

for epoch in range(1, Config.epochs + 1):
    avg_loss, avg_clf, avg_kl = train_minibatch(epoch)
    scheduler.step()
    
    # 获取 Sparsity 参考
    model.eval()
    with torch.no_grad():
        sample_batch = next(iter(train_loader)).to(device)
        _, weights, _ = model(sample_batch.x, sample_batch.edge_index)
        avg_sp = weights.mean().item()

    if epoch % 5 == 0: 
        valid_acc = inference_minibatch(valid_loader)
        test_acc = inference_minibatch(test_loader)
        
        if valid_acc > best_valid_acc:
            best_valid_acc = valid_acc
            stagnant_epochs = 0
            torch.save(model.state_dict(), 'best_gsat_products.pt')
            status = "[*] NEW BEST"
        else:
            stagnant_epochs += 1
            status = "[ ]"
            
        print(f"\n{status} Ep: {epoch:03d} | Val:{valid_acc:.4f} | Test:{test_acc:.4f} | KL_L:{avg_kl:.4f} | SP:{avg_sp:.4f}")
        
        if stagnant_epochs >= Config.patience: 
            print(f"\n[Early Stop] at epoch {epoch}")
            break
        torch.cuda.empty_cache()
    else:
        print(f"      Ep: {epoch:03d} | Clf_L: {avg_clf:.4f} | KL_L: {avg_kl:.4f} | SP: {avg_sp:.4f}", end='\r')

/home/malina/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/malina/.venv/lib/python3.8/site-packages/ogb/nodeproppred/dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. 

Starting GSAT on Products: Beta: 0.3, R: 0.3
      Ep: 004 | Clf_L: 0.4055 | KL_L: 0.0092 | SP: 0.0153
[*] NEW BEST Ep: 005 | Val:0.8820 | Test:0.7385 | KL_L:0.0093 | SP:0.0154
      Ep: 009 | Clf_L: 0.3950 | KL_L: 0.0098 | SP: 0.0165
[*] NEW BEST Ep: 010 | Val:0.8825 | Test:0.7245 | KL_L:0.0097 | SP:0.0164
      Ep: 014 | Clf_L: 0.3729 | KL_L: 0.0106 | SP: 0.0166
[*] NEW BEST Ep: 015 | Val:0.8893 | Test:0.7529 | KL_L:0.0107 | SP:0.0174
      Ep: 019 | Clf_L: 0.3615 | KL_L: 0.0106 | SP: 0.0174
[*] NEW BEST Ep: 020 | Val:0.8907 | Test:0.7458 | KL_L:0.0111 | SP:0.0179
      Ep: 024 | Clf_L: 0.3466 | KL_L: 0.0117 | SP: 0.0200
[*] NEW BEST Ep: 025 | Val:0.8952 | Test:0.7528 | KL_L:0.0114 | SP:0.0191
      Ep: 029 | Clf_L: 0.3407 | KL_L: 0.0121 | SP: 0.0183
[ ] Ep: 030 | Val:0.8948 | Test:0.7569 | KL_L:0.0119 | SP:0.0187
      Ep: 034 | Clf_L: 0.3194 | KL_L: 0.0129 | SP: 0.0208
[*] NEW BEST Ep: 035 | Val:0.8986 | Test:0.7626 | KL_L:0.0132 | SP:0.0193
      Ep: 039 | Clf_L: 0.3178 | KL_L: 0.

KeyboardInterrupt: 